# Some Semantic Grouping Analysis of the FAIR-CARE Survey Text Responses

In [1]:
from IPython.display import display

import copy
import matplotlib.pyplot as plt

from openpyxl import Workbook

import os

import numpy as np
import pandas as pd

from slugify import slugify

# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
# Use this path to save the wordcloud outputs

col_config_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'imls-fair-care-survey-columns-config.csv',
)

processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
df_config = pd.read_csv(col_config_path, low_memory=False)

print(f'FAIR+CARE survey has {len(df.index)} rows')
print(f'FAIR+CARE survey config has {len(df_config.index)} rows')

FAIR+CARE survey has 787 rows
FAIR+CARE survey token extract has 37179 rows
FAIR+CARE survey ngram extract has 216860 rows
FAIR+CARE survey semantic groupings has 6128 rows


In [2]:
gen_group_cols = ['orig_column_index', 'column', semantic_group_col,]
use_index = df_semantic['orig_column_index'] >= 38

# Count up the number of survey responses for each semantic group in each column.
df_semantic_gp1 = df_semantic[use_index ][(['Response ID'] + gen_group_cols)].groupby(gen_group_cols, as_index=False).agg({'Response ID': 'nunique'})
# Rename the count of survey responses in each column's semantic groups.
df_semantic_gp1.rename(columns={'Response ID': 'count_group_responses'}, inplace=True)
df_semantic_gp1.sort_values(by=['orig_column_index', 'count_group_responses', semantic_group_col,], ascending=[True, False, True], inplace=True)


# Repeatly used ngrams
print(f'There are {len(df_semantic_gp1.index)} unique semantic groups across all text fields')

df_semantic_gp1.head(100)





There are 127 unique semantic groups across all text fields


,orig_column_index,column,semantic_cluster_id,count_group_responses
0,38,"Data Role: How You Work with Data, Text",0.0,16
1,48,"Data Org: Other, Text",0.0,19
2,48,"Data Org: Other, Text",1.0,4
3,50,"Data Store: Data Storage, Other, Text",0.0,4
4,50,"Data Store: Data Storage, Other, Text",1.0,4
...,...,...,...,...
93,149,"Do You Attribute: Yes, No, Text",1.0,5
94,149,"Do You Attribute: Yes, No, Text",2.0,5
96,149,"Do You Attribute: Yes, No, Text",4.0,5
97,149,"Do You Attribute: Yes, No, Text",5.0,4


In [3]:
def get_original_column_name(orig_column_index, df_config=df_config):
    """Gets the original column name from the df_config (column configuration data)"""
    config_index = (df_config['orig_column_index'] == orig_column_index)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    return row['raw_column']


def get_column_total_nonblank_response_count(col, df=df):
    """Gets the total number of nonblank reponses to a specific column"""
    if not col in df.columns.tolist():
        return None
    col_index = ~df[col].isnull()
    return len(df[col_index].index)


# Now Make WordClouds for each column, and each cluster
def make_wordcloud_text_dict_from_df_wc(act_filter, df_wc):
    """Makes a wordcloud text dict keyed by token, with occurance counts from an index and dataframe"""
    df_wc_grp = df_wc[act_filter][['token', 'token_count']].groupby(
        ['token',], 
        as_index=False
    ).agg(
        {
            'token': 'first',
            'token_count': 'sum',
        }
    )
    df_wc_grp.sort_values(by=['token_count', 'token'], ascending=[False, True], inplace=True)
    df_wc_grp_head = df_wc_grp.head(500)
    text_dict = {}
    df_wc_grp_head = df_wc_grp.head(500)
    for _, row in df_wc_grp_head.iterrows():
        if row['token'] == 'data':
            # not very interesting
            continue
        text_dict[row['token']] = row['token_count']
    return text_dict


# Now Make WordClouds for each column, and each cluster
def create_wordcloud(text_dict, title=None, caption=None, file_suffix='', save_dir=wordcloud_path):
    plt.rcParams["figure.figsize"] = (6, 6)
    wc = WordCloud(background_color="white", max_words=500, width=500, height=300)
    wc.generate_from_frequencies(text_dict)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    if title:
        plt.title(title)
    if caption:
        plt.figtext(0.5, 0.01, caption, wrap=True, horizontalalignment='center', fontsize=9)
    slug_file = slugify(title)
    plt.autoscale()
    filename = f'{slug_file}{file_suffix}.png'
    f_path = os.path.join(save_dir, filename)
    plt.savefig(f_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f'Saved figure: {filename}')


token_join_cols = ['Response ID', 'token', 'token_count', 'orig_column_index',]
semantic_join_cols = [
    'Response ID', 
    'orig_column_index', 
    'Question Number', 
    'Section', 
    'Sub-Section', 
    'column', 
    'response', 
    semantic_group_col,
    'semantic_agglom_cluster_id',
    'bert_name_plain', 
    'bert_top_words_plain',
]
df_wc_pre = pd.merge(df_token[token_join_cols], df_semantic[semantic_join_cols], on=['Response ID', 'orig_column_index',])
# Merge in the count of responses in each column's semantic groups
gp1_join_cols = ['orig_column_index', semantic_group_col, 'count_group_responses',]
df_wc = pd.merge(df_wc_pre, df_semantic_gp1[gp1_join_cols], on=['orig_column_index', semantic_group_col])


In [4]:
use_index = df_wc['orig_column_index'] >= 38
df_wc_grp = df_wc[use_index].groupby(
    ['orig_column_index', 'column', semantic_group_col, 'token',], 
    as_index=False
).agg(
    {
        'Question Number': 'first',
        'Section': 'first',
        'Sub-Section': 'first',
        'token_count': 'sum',
        'count_group_responses': 'first',
    }
)

for col in df_wc_grp['column'].unique().tolist():
    col_index = df_wc_grp['column'] == col
    orig_column_index = df_wc_grp[col_index]['orig_column_index'].iloc[0]
    col_allgroup_sum_count = get_column_total_nonblank_response_count(col)
    sm_groups = df_wc_grp[col_index][semantic_group_col].unique().tolist()
    for sm_group in sm_groups:
        col_sm_index = col_index & ( df_wc_grp[semantic_group_col] == sm_group)
        sematic_group_resp_count = df_wc_grp[col_sm_index]['count_group_responses'].iloc[0]
        total_kw_count = df_wc_grp[col_sm_index]['count_group_responses'].sum()
        sum_act_tokens = df_wc_grp[col_sm_index]['token_count'].sum()
        question = df_wc_grp[col_sm_index]['Question Number'].iloc[0]
        question = int(question)
        section = df_wc_grp[col_sm_index]['Section'].iloc[0]
        sub_section = df_wc_grp[col_sm_index]['Sub-Section'].iloc[0]
        orig_col_name = get_original_column_name(orig_column_index)
        t_sm_group = sm_group
        try:
            # turn the sm_group ID into an integrer value
            t_sm_group = int(sm_group)
        except:
            pass
        title = f'{section} -- {sub_section} ({question}) {col}\n[Group {t_sm_group}]'
        caption = f'[Question: {question}] "{orig_col_name}"\n(Group includes n = {sematic_group_resp_count} responses; question had {col_allgroup_sum_count} total responses)'
        # Make the first wordcloud for tokens used by this current semantic group.
        text_dict = make_wordcloud_text_dict_from_df_wc(act_filter=col_sm_index, df_wc=df_wc_grp)
        create_wordcloud(text_dict, title, caption)
        # Now make a filter for all the tokens used in a given question
        all_act_filter = col_index
        sum_all_tokens = df_wc_grp[all_act_filter]['token_count'].sum()
        all_text_dict = make_wordcloud_text_dict_from_df_wc(all_act_filter, df_wc_grp)
        filtered_text_dict = {}
        for token, count in text_dict.items():
            act_token_rate = count / sum_act_tokens
            all_token_rate = all_text_dict.get(token, 0) / sum_all_tokens
            # the expected count for this token is derived from the relative
            # frequency of this token from the unfiltered set of responses
            expected_count = sum_act_tokens * all_token_rate
            count_dif_expected = count - expected_count
            if count_dif_expected <= 1:
                # The difference from the expected count is less than one,
                # so don't display this token in a wordcloud
                continue
            count_dif_expected = int(round(count_dif_expected, 0))
            filtered_text_dict[token] = count_dif_expected
        if not filtered_text_dict:
            # None of the tokens fit our criteria for making a filtered, wordcloud.
            continue
        title += ' [Reweighted for Distinctive]'
        filtered_file_suffix = '-weighted'
        create_wordcloud(filtered_text_dict, title, caption, file_suffix=filtered_file_suffix)



Saved figure: demographics-data-role-1-data-role-how-you-work-with-data-text-group-0.png
Saved figure: fair-findable-2-data-org-other-text-group-0.png
Saved figure: fair-findable-2-data-org-other-text-group-1.png
Saved figure: fair-findable-3-data-store-data-storage-other-text-group-0.png
Saved figure: fair-findable-3-data-store-data-storage-other-text-group-0-reweighted-for-distinctive-weighted.png
Saved figure: fair-findable-3-data-store-data-storage-other-text-group-1.png
Saved figure: fair-findable-4-findable-yes-text-group-0.png
Saved figure: fair-findable-4-findable-yes-text-group-0-reweighted-for-distinctive-weighted.png
Saved figure: fair-findable-4-findable-yes-text-group-1.png
Saved figure: fair-findable-4-findable-yes-text-group-1-reweighted-for-distinctive-weighted.png
Saved figure: fair-findable-4-findable-yes-text-group-2.png
Saved figure: fair-findable-4-findable-yes-text-group-2-reweighted-for-distinctive-weighted.png
Saved figure: fair-findable-4-findable-yes-text-grou